In [1]:
from pathlib import Path
import sys

import torch
from torch.utils.data import DataLoader

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from dataset import PatchDataset
from inference import MicroLadPredictor
from models import CustomVAE, DDPM, TimeUNet

DATA_DIR = ROOT / "data" / "images"
VAE_CKPT = ROOT / "microlad-anode" / "vae_anode.pth"
PATCH_SIZE = 64
TIMESTEPS = 3
AXIS = "z"
AXIS_ID = 0
SLICE_INDEX = 12
SEED = 0


In [2]:
dataset = PatchDataset(DATA_DIR, patch_size=PATCH_SIZE, seed=SEED)
batch = next(iter(DataLoader(dataset, batch_size=1, shuffle=False)))
condition = batch
condition.shape, AXIS_ID, SLICE_INDEX


(torch.Size([1, 1, 64, 64]), 0, 12)

In [3]:
vae = CustomVAE(latent_ch=4)
vae.load_state_dict(torch.load(VAE_CKPT, map_location="cpu")["vae"])
vae.eval()

unet = TimeUNet(latent_ch=4, base_ch=16, time_dim=16)
unet.eval()
ddpm = DDPM(timesteps=TIMESTEPS)


In [4]:
predictor = MicroLadPredictor(vae=vae, unet=unet, ddpm=ddpm)
result = predictor.predict({"images": [{"image": condition, "axis": AXIS_ID, "index": SLICE_INDEX}]})
volume = result["volume"]
locked_delta = (volume[SLICE_INDEX] - condition.squeeze(0)).abs().max()

volume.shape, float(locked_delta)


(torch.Size([4, 16, 16, 16]),
 torch.Size([64, 1, 64, 64]),
 torch.Size([1, 4, 16, 16]),
 3,
 0.0)